In [ ]:
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples
import numpy as np

In [ ]:
import networkx as nx
from functools import reduce
from qiskit.quantum_info import SparsePauliOp

# THIS IS THE CORRECT SETTING FOR READING LSB 
# NOT GOING TO CHANGE 
def indices_to_pauli(t: int, k: int, n: int, T: int):
    p = ['I'] * n * T
    p[t*n + k] = 'Z'
    return SparsePauliOp(''.join(p)[::-1], np.array([1]))

def bin_rep(k, n):
    return [int(x) for x in np.binary_repr(k, n)]

rng = np.random.default_rng()

In [72]:
def graph_to_hubo_hamiltonian(
        graph: nx.Graph, n: int, T: int, lamda: float, constraint_terms: float | tuple[int,...]=1.0
) -> tuple[SparsePauliOp, float]:
    nodes = list(graph.nodes)
    V = len(nodes)
    if isinstance(constraint_terms, tuple):
        terms_to_keep = constraint_terms
    elif isinstance(constraint_terms, float):
        terms_to_keep = rng.choice(T-1, int(np.ceil((T-1) * constraint_terms)), replace=False)
    else:
        raise Exception(f'Expected float or tuple of ints for constraint_terms, got {constraint_terms}')
    print(f'Keeping constraints at times: {terms_to_keep}')   
    cons_spo = reduce(
        SparsePauliOp._add,
        [
            SparsePauliOp('I'*n*T, np.array([1])) - reduce(
                SparsePauliOp._add,
                [
                    SparsePauliOp.compose(
                        reduce(
                            SparsePauliOp.compose,
                            [0.5 * (SparsePauliOp('I' * n * T, np.array([1])) + (1 - 2 * bin_rep(i, n)[k]) * indices_to_pauli(t, k, n, T)) for k in range(n)],
                            SparsePauliOp('I'*n*T, np.array([1]))
                        ),
                        reduce(
                            SparsePauliOp._add,
                            [
                                reduce(
                                    SparsePauliOp.compose,
                                    [0.5 * (SparsePauliOp('I' * n * T, np.array([1])) + (1 - 2 * bin_rep(j, n)[k]) * indices_to_pauli(t+1, k, n, T)) for k in range(n)],
                                    SparsePauliOp('I'*n*T, np.array([1]))
                                )
                                for j in [nodes.index(nbr) for nbr in graph.neighbors(nodes[i])]
                            ],
                            SparsePauliOp('I'*n*T, np.array([0]))
                        )
                    )
                    for i in range(V)
                ],
                SparsePauliOp('I'*n*T, np.array([0]))
            ) for t in terms_to_keep
        ],
        SparsePauliOp('I'*n*T, np.array([0]))
    )


    obj_spo = reduce(
        SparsePauliOp._add,
        [
            (
                reduce(
                    SparsePauliOp._add,
                    [
                        reduce(
                            SparsePauliOp.compose,
                            [0.5 * (SparsePauliOp('I' * n * T, np.array([1])) + (1 - 2 * bin_rep(i, n)[k]) * indices_to_pauli(t, k, n, T)) for k in range(n)]
                        ) + reduce(
                            SparsePauliOp.compose,
                            [0.5 * (SparsePauliOp('I' * n * T, np.array([1])) + (1 - 2 * bin_rep(i+1, n)[k]) * indices_to_pauli(t, k, n, T)) for k in range(n)]
                        )
                        for t in range(T)
                    ],
                    SparsePauliOp('I'  * n * T, np.array([0]))
                ) 
                - SparsePauliOp('I' * n * T, graph.nodes[nodes[i]]["weight"]) 
            ) ** 2
            for i in range(0, V, 2)
        ],
        SparsePauliOp('I'  * n * T, np.array([0]))
    )

    hamiltonian = lamda * cons_spo + obj_spo
    hamiltonian = hamiltonian.simplify()
    hamiltonian = hamiltonian.sort()
    norm = np.abs(max(hamiltonian.coeffs[1:]))
    hamiltonian /= norm
    return hamiltonian, norm

In [73]:
filename = 'test_N2_W2'
copy_numbers = [1., 1.]
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
full_hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
num_qubits = n * T
keys = [np.binary_repr(x, num_qubits) for x in range(2**num_qubits)]
evals = np.round(evaluate_sparse_pauli_samples(keys, full_hamiltonian * norm), 3)

Keeping constraints at times: [0]


u0+ 00
u0- 01
u1+ 10
u1- 11
"Read time left to right, nodes left to right"

In [74]:
evals

array([12., 10., 12., 10.,  0., 12., 10., 12., 12., 10., 12.,  0., 10.,
       12., 10., 12.])

In [76]:
evals[int('0100', 2)], evals[int('1011', 2)]

(np.float64(0.0), np.float64(0.0))

In [77]:
'0100'[::-1], '1011'[::-1]

('0010', '1101')

In [78]:
filename = 'trivial'
copy_numbers = [1., 1.,1.]
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
full_hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
num_qubits = n * T

Keeping constraints at times: [1 0]


In [79]:
num_qubits

9

u0+ 000
u0- 001
u1+ 010
u1- 011
u2+ 100
u2- 101
"Read time left to right, nodes right to left"

In [80]:
keys = [np.binary_repr(x, num_qubits) for x in range(2**num_qubits)]

evals = np.round(evaluate_sparse_pauli_samples(keys, full_hamiltonian * norm), 3)

In [83]:
evals[int('001010000', 2)], evals[int('100110101', 2)]

(np.float64(0.0), np.float64(0.0))